In [ ]:
import platform
import kagglehub as kh
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split , KFold , cross_validate , RandomizedSearchCV
from IPython.display import display
from helper_utils import data_split
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.linear_model import LinearRegression , Ridge , Lasso , ElasticNet 
from sklearn.base import BaseEstimator , TransformerMixin
from sklearn.ensemble import ExtraTreesRegressor , RandomForestRegressor
from sklearn.pipeline import Pipeline
import joblib
import os
from pathlib import Path
from scipy.stats import randint , uniform , loguniform
from AmesFeatures import *
from sklearn.metrics import r2_score , mean_absolute_error , root_mean_squared_error , mean_squared_error

pd.set_option('display.max_rows',None)
display(platform.python_version())

'3.14.4'

In [29]:
TARGET='SalePrice'
data_path=Path('./data/AmesHousing.csv')
if not data_path.exists():
    print("Data not found..\n Downloading the dataset")
    path=kh.dataset_download("shashanknecrothapa/ames-housing-dataset",output_dir="./data",force_download=True)

raw = pd.read_csv(data_path)
if TARGET not in raw:
    raise ValueError(f"Expected target {TARGET!r} was not found.")
if raw[TARGET].isna().any():
    raise ValueError("Target contains missing values; define a target policy before modeling.")

audit = pd.Series({
    "Rows": len(raw),
    "Columns": raw.shape[1],
    "Numeric columns": raw.select_dtypes(include=np.number).shape[1],
    "Categorical columns": raw.select_dtypes(exclude=np.number).shape[1],
    "Exact duplicate rows": int(raw.duplicated().sum()),
    "Missing targets": int(raw[TARGET].isna().sum()),
}, name="value")
display(audit.to_frame())
display(raw.head(10))

,value
Rows,2930
Columns,82
Numeric columns,39
Categorical columns,43
Exact duplicate rows,0
Missing targets,0


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,...,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,5,1960,1960,Hip,CompShg,BrkFace,Plywood,Stone,112.0,TA,TA,CBlock,TA,Gd,Gd,BLQ,639.0,Unf,0.0,441.0,1080.0,...,Y,SBrkr,1656,0,0,1656,1.0,0.0,1,0,3,1,TA,7,Typ,2,Gd,Attchd,1960.0,Fin,2.0,528.0,TA,TA,P,210,62,0,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Rec,468.0,LwQ,144.0,270.0,882.0,...,Y,SBrkr,896,0,0,896,0.0,0.0,1,0,2,1,TA,5,Typ,0,NaN,Attchd,1961.0,Unf,1.0,730.0,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0,TA,TA,CBlock,TA,TA,No,ALQ,923.0,Unf,0.0,406.0,1329.0,...,Y,SBrkr,1329,0,0,1329,0.0,0.0,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,1958.0,Unf,1.0,312.0,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,7,5,1968,1968,Hip,CompShg,BrkFace,BrkFace,NaN,0.0,Gd,TA,CBlock,TA,TA,No,ALQ,1065.0,Unf,0.0,1045.0,2110.0,...,Y,SBrkr,2110,0,0,2110,1.0,0.0,2,1,3,1,Ex,8,Typ,2,TA,Attchd,1968.0,Fin,2.0,522.0,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,PConc,Gd,TA,No,GLQ,791.0,Unf,0.0,137.0,928.0,...,Y,SBrkr,928,701,0,1629,0.0,0.0,2,1,3,1,TA,6,Typ,1,TA,Attchd,1997.0,Fin,2.0,482.0,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900
5,6,527105030,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,6,1998,1998,Gable,CompShg,VinylSd,VinylSd,BrkFace,20.0,TA,TA,PConc,TA,TA,No,GLQ,602.0,Unf,0.0,324.0,926.0,...,Y,SBrkr,926,678,0,1604,0.0,0.0,2,1,3,1,Gd,7,Typ,1,Gd,Attchd,1998.0,Fin,2.0,470.0,TA,TA,Y,360,36,0,0,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal,195500
6,7,527127150,120,RL,41.0,4920,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,StoneBr,Norm,Norm,TwnhsE,1Story,8,5,2001,2001,Gable,CompShg,CemntBd,CmentBd,NaN,0.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,616.0,Unf,0.0,722.0,1338.0,...,Y,SBrkr,1338,0,0,1338,1.0,0.0,2,0,2,1,Gd,6,Typ,0,NaN,Attchd,2001.0,Fin,2.0,582.0,TA,TA,Y,0,0,170,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,213500
7,8,527145080,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,Inside,Gtl,StoneBr,Norm,Norm,TwnhsE,1Story,8,5,1992,1992,Gable,CompShg,HdBoard,HdBoard,NaN,0.0,Gd,TA,PConc,Gd,TA,No,ALQ,263.0,Unf,0.0,1017.0,1280.0,...,Y,SBrkr,1280,0,0,1280,0.0,0.0,2,0,2,1,Gd,5,Typ,0,NaN,Attchd,1992.0,RFn,2.0,506.0,TA,TA,Y,0,82,0,0,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal,191500
8,9,527146030,120,RL,39.0,5389,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,StoneBr,Norm,Norm,TwnhsE,1Story,8,5,1995,1996,Gable,CompShg,CemntBd,CmentBd,NaN,0.0,Gd,TA,PConc,Gd,TA,No,GLQ,1180.0,Unf,0.0,415.0,1595.0,...,Y,SBrkr,1616,0,0,1616,1.0,0.0,2,0,2,1,Gd,5,Typ,1,TA,Attchd,1995.0,RFn,2.0,6

In [30]:
X_train,X_test,Y_Train,Y_Test=data_split(raw)

In [ ]:
# Drop the well-documented Ames anomalies (huge homes that sold far below
# trend) from the TRAINING set only. Never touch X_test/Y_Test this way -
# that would silently inflate reported test metrics.
X_train, Y_Train = drop_known_outliers(X_train, Y_Train)
print(f"Training rows after outlier removal: {len(X_train)}")

In [58]:
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error"
}

estimators={
    'Lasso':{
        'model':Lasso(),
        'parameters':{
            'regressor__model__alpha':loguniform(a=0.0001, b=1.0),
            "regressor__model__max_iter": [5000, 10000, 20000]
        }
    },
    'Ridge':{
        'model':Ridge(),
        'parameters' : {
            'regressor__model__alpha':loguniform(a=0.0001, b=1.0),
            "regressor__model__max_iter": [5000, 10000, 20000]
        }
    },
    'ElasticNet' :{
        'model':ElasticNet(),
        'parameters':{
            'regressor__model__alpha':loguniform(a=0.0001, b=1.0),
            'regressor__model__l1_ratio':loguniform(a=0.0001, b=1.0),
            "regressor__model__max_iter": [5000, 10000, 20000]
        }
    },
    'ExtraTreesRegressor':{
        'model':ExtraTreesRegressor(random_state=42),
        'parameters':{
            "regressor__model__n_estimators": [100, 200, 300, 500],
            "regressor__model__max_depth": [None, 5, 10, 20, 30, 50],
            "regressor__model__min_samples_split": [2, 5, 10, 20],
            "regressor__model__min_samples_leaf": [1, 2, 4, 8],
            "regressor__model__max_features": [1.0, "sqrt", "log2"],
            "regressor__model__bootstrap": [True, False]
        }
    },
    'RandomForestRegressor':{
        'model':RandomForestRegressor(random_state=42),
        'parameters':{
            "regressor__model__n_estimators": [100, 200, 300, 500],
            "regressor__model__max_depth": [None, 5, 10, 20, 30, 50],
            "regressor__model__min_samples_split": [2, 5, 10, 20],
            "regressor__model__min_samples_leaf": [1, 2, 4, 8],
            "regressor__model__max_features": [1.0, "sqrt", "log2"],
            "regressor__model__bootstrap": [True, False]
        }
    }

}

In [59]:
# forming a baseline with LinearRegression
model=build_model_pipeline(LinearRegression(),scale_numeric=True,use_log_target=True)
cv=KFold(n_splits=5,shuffle=True,random_state=42)
results=cross_validate(
    estimator=model,
    X=X_train,
    y=Y_Train,
    cv=cv,
    scoring=scoring
)
pd.DataFrame(
    {
        'Training time':[results['fit_time'].mean(0)],
        'Validating time':[results['score_time'].mean(0)],
        'R2 score':[results['test_r2'].mean(0)],
        'MSE score':[results['test_mse'].mean(0)],
        'MAE score':[results['test_mae'].mean(0)],
        'RMSE score':[results['test_rmse'].mean(0)]
    }
)

,Training time,Validating time,R2 score,MSE score,MAE score,RMSE score
0,0.28019,0.025551,0.833399,-9.984786e+08,-14244.618737,-27967.844934


In [65]:
models={}
for i in estimators.keys():
    search=RandomizedSearchCV(
        estimator=build_model_pipeline(estimators[i]['model'],scale_numeric=True,use_log_target=True),
        param_distributions=estimators[i]['parameters'],
        n_iter=30,
        cv=5,
        scoring=scoring,
        refit="rmse",
        random_state=42,
        n_jobs=-1
    )
    search.fit(X_train,Y_Train)
    models[i]=(search.best_estimator_)


/home/trozan24/Downloads/Coding/projects/Ames-House-Prediction/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.578654e+00, tolerance: 3.013e-02
  model = cd_fast.enet_coordinate_descent(
/home/trozan24/Downloads/Coding/projects/Ames-House-Prediction/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.601120e+00, tolerance: 2.998e-02
  model = cd_fast.enet_coordinate_descent(
/home/trozan24/Downloads/Coding/projects/Ames-House-Prediction/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did

In [71]:
data={
    'model':[],
    'RMSE score':[],
    'MSE score':[],
    'MAE score':[],
    'R2 score':[]
}

# performance for train score
for i in models.keys():
    x=X_train
    y=Y_Train
    y_pred=models[i].predict(x)
    data['model'].append(i)
    data['R2 score'].append(r2_score(y,y_pred))
    data['RMSE score'].append(root_mean_squared_error(y,y_pred))
    data['MSE score'].append(mean_squared_error(y,y_pred))
    data['MAE score'].append(mean_absolute_error(y,y_pred))

train_statistics=pd.DataFrame(data)
display(train_statistics)

,model,RMSE score,MSE score,MAE score,R2 score
0,Lasso,21301.980422,4.537744e+08,12949.881051,0.923681
1,Ridge,18484.074110,3.416610e+08,12158.551764,0.942537
2,ElasticNet,19201.185570,3.686855e+08,12335.604780,0.937992
3,ExtraTreesRegressor,2961.149430,8.768406e+06,1525.195983,0.998525
4,RandomForestRegressor,13494.165620,1.820925e+08,8634.527993,0.969374


In [72]:
data={
    'model':[],
    'RMSE score':[],
    'MSE score':[],
    'MAE score':[],
    'R2 score':[]
}

# performance for train score
for i in models.keys():
    x=X_test
    y=Y_Test
    y_pred=models[i].predict(x)
    data['model'].append(i)
    data['R2 score'].append(r2_score(y,y_pred))
    data['RMSE score'].append(root_mean_squared_error(y,y_pred))
    data['MSE score'].append(mean_squared_error(y,y_pred))
    data['MAE score'].append(mean_absolute_error(y,y_pred))

test_statistics=pd.DataFrame(data)
display(test_statistics)

,model,RMSE score,MSE score,MAE score,R2 score
0,Lasso,28647.504465,8.206795e+08,14903.625991,0.897640
1,Ridge,28038.451552,7.861548e+08,14728.199052,0.901946
2,ElasticNet,28112.392148,7.903066e+08,14800.523517,0.901428
3,ExtraTreesRegressor,24228.374392,5.870141e+08,14785.064963,0.926784
4,RandomForestRegressor,26369.943069,6.953739e+08,15937.213894,0.913269


In [ ]:
# Pick the best model by test RMSE and save it for deployment.
# model.pkl is a single self-contained sklearn Pipeline: feature engineering,
# encoding, scaling, and the trained model all in one object. A frontend only
# needs this file + AmesFeatures.py (for the AmesFeatureEngineer class) and
# can call model.predict(raw_dataframe) directly - no separate preprocessing
# step needed on the frontend side.
best_name = test_statistics.loc[test_statistics['RMSE score'].idxmin(), 'model']
best_model = models[best_name]
print(f"Best model on test RMSE: {best_name}")

joblib.dump(best_model, 'model.pkl')
print("Saved model.pkl")

## Model explainability (SHAP)

`shap.Explainer` auto-detects the underlying model type (linear, tree, etc.)
and picks an appropriate exact/fast explainer, so this works regardless of
which model won the comparison above.

In [ ]:
import shap

fitted_pipeline = best_model.regressor_  # inner Pipeline inside TransformedTargetRegressor
preprocessing = fitted_pipeline.named_steps['preprocessing']
inner_model = fitted_pipeline.named_steps['model']

feature_names = preprocessing.named_steps['column_transform'].get_feature_names_out()
feature_names = [f.split('__', 1)[-1] for f in feature_names]  # strip ColumnTransformer prefixes

X_train_transformed = preprocessing.transform(X_train)
X_test_transformed = preprocessing.transform(X_test)

background = shap.sample(X_train_transformed, 100, random_state=42)
explainer = shap.Explainer(inner_model, background, feature_names=feature_names)
shap_values = explainer(X_test_transformed[:100])

**Global**: which features move predictions most, across 100 test houses.

In [ ]:
shap.summary_plot(shap_values, X_test_transformed[:100], feature_names=feature_names, max_display=15)

**Local**: why the model predicted what it did for one specific house.

In [ ]:
shap.plots.waterfall(shap_values[0], max_display=12)